In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
ROOT_DIR = Path.cwd().parent

data_path = ROOT_DIR / "data"

In [3]:
ur = pd.read_csv(data_path / "underlyings_reference.csv")
dv = pd.read_csv(data_path / "daily_volatility.csv")

In [4]:
display(ur)

,underlying,sector,structural_base_vol
0,KYBR,Mineria de materiales estrategicos,0.22
1,CORL,Logistica y transporte,0.28
2,MNDO,Defensa y armamento,0.35
3,HTTX,Comercio y materias primas,0.50
4,TECH,Tecnologia y automatizacion,0.32
5,SITH,Energia e infraestructuras,0.26
6,JEDI,Gestion patrimonial,0.14
7,BSKR,Metales y aleaciones,0.30
8,POBK,Ocio y entretenimiento,0.42
9,NABO,Agricultura y biotecnologia,0.11


- `REBL` (Renta fija especulativa) mostraba correlación **negativa** con el resto de subyacentes, coherente con la relación típica entre renta fija y renta variable. Cuando la volatilidad de la renta variable tiende a subir, la dinámica de tipos puede moverse en sentido contrario.

In [5]:
ur['sector'].nunique()

14

Hay 14 sectores distintos, uno por subyacente. Esto quiere decir que a efectos prácticos, sector y subyacente son la misma variable.
- **Limitación:** Con esta granularidad de datos la propuesta de feature `n_dif_sectors` (número de sectores distintos en la cesta), se vuelve inecesaria, de hecho se convierte en la misma información de `n_underlyings`.

In [6]:
display(ur.info())
display(ur.describe())

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   underlying           14 non-null     str    
 1   sector               14 non-null     str    
 2   structural_base_vol  14 non-null     float64
dtypes: float64(1), str(2)
memory usage: 468.0 bytes


None

,structural_base_vol
count,14.000000
mean,0.272143
std,0.108994
min,0.110000
25%,0.190000
50%,0.275000
75%,0.317500
max,0.500000


La volatilidad es una métrica estrictamente positiva, `structural_base_vol` cumple la cota esperada.

In [7]:
vol_media_realizada = dv.groupby('underlying')['realized_vol_63d'].mean()
comparacion = pd.DataFrame({
    'structural_base_vol': ur.set_index('underlying')['structural_base_vol'],
    'vol_media_realizada': vol_media_realizada
})
comparacion['ratio'] = comparacion['vol_media_realizada'] / comparacion['structural_base_vol']
comparacion.sort_values('vol_media_realizada', ascending=False)

,structural_base_vol,vol_media_realizada,ratio
underlying,,,
HTTX,0.50,0.874216,1.748432
POBK,0.42,0.507089,1.207355
REBL,0.18,0.477894,2.654965
MNDO,0.35,0.429858,1.228167
TECH,0.32,0.391437,1.223242
DRC,0.31,0.381730,1.231386
BSKR,0.30,0.364846,1.216152
CORL,0.28,0.345805,1.235020
WOOK,0.27,0.332874,1.232868


`REBL` tiene un ratio de 2.65, más del doble que cualquier otro subyacente, exceptuando a `HTTX` con 1.75. Es decir, `structural_base_vol` para  `REBL` (0.18) es notablemente más bajo de o que debería ser según su propia volatilidad realizada media (0.478).